In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def load():
    demo = pd.read_sas("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.xpt")
    bp = pd.read_sas("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BPX_J.xpt")
    chol = pd.read_sas("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/TCHOL_J.xpt")
    return demo, bp, chol

def info(demo, bp, chol):
    print(f'Demographics: {demo.shape} with columns {demo.columns}')
    print(f'Blood Pressure: {bp.shape} with columns {bp.columns}')
    print(f'Cholesterol: {chol.shape} with columns {chol.columns}')

def merge(demo, bp, chol):
    columns = ["SEQN", "RIAGENDR", "RIDAGEYR", "RIDRETH3", "RIDEXPRG", "BPXSY1", "BPXDI1", "BPXSY2", "BPXDI2", "BPXSY3", "BPXDI3", "BPXSY4", "BPXDI4", "LBXTC"]
    merged = demo.merge(bp, on="SEQN", how="left").merge(chol, on="SEQN", how="left")[columns]
    return merged

def merge_bp(merged):
    bpcolumns = ['BPXSY1', 'BPXSY2', 'BPXSY3', 'BPXSY4', 'BPXDI1', 'BPXDI2', 'BPXDI3', 'BPXDI4']
    merged['SBP Merged'] = merged[['BPXSY1', 'BPXSY2', 'BPXSY3', 'BPXSY4']].mean(axis=1)
    merged['DBP Merged'] = merged[['BPXDI1', 'BPXDI2', 'BPXDI3', 'BPXDI4']].mean(axis=1)
    return merged

def reorg(merged):
    merged.rename(columns={"SEQN": "ID Number",
            "RIAGENDR": "Gender",
            "RIDAGEYR": "Age",
            "RIDRETH3": "Race-Ethnicity",
            "RIDEXPRG": "Pregnancy Status",
            "BPXSY1": "Systolic BP, Trial 1",
            "BPXDI1": "Diastolic BP, Trial 1",
            "BPXSY2": "Systolic BP, Trial 2",
            "BPXDI2": "Diastolic BP, Trial 2", 
            "BPXSY3": "Systolic BP, Trial 3",
            "BPXDI3": "Diastolic BP, Trial 3", 
            "BPXSY4": "Systolic BP, Trial 4",
            "BPXDI4": "Diastolic BP, Trial 4", 
            "LBXTC": "Total Cholesterol (mg/dl)"}, inplace = True)
    merged['Gender'] = merged['Gender'].replace({1: "M", 2: "F"})
    merged["Race-Ethnicity"] = merged['Race-Ethnicity'].replace({1: "Mexican American", 2: "Other Hispanic", 3: "White", 4: "Black", 6: "Asian", 7: "Other or Multiracial"})
    merged["Pregnancy Status"] = merged["Pregnancy Status"].replace({1: "Pregnant", 2: "Not Pregnant", 3: "Unknown"}).fillna("Unknown")
    merged = merged.round(2)
    return merged

def age_filtering(merged):
    n_before = len(merged)
    merged = merged[merged["Age"] >= 18].copy()
    print(f"Excluded {n_before - len(merged)} minors. Analytic sample: n={len(merged):,}")
    return merged

def bp_grouping(merged):
    sbp = merged["SBP Merged"].values
    dbp = merged["DBP Merged"].values
    conditions1 = [(sbp < 120) & (dbp < 80), 
                  (sbp >= 120) & (sbp < 129) & (dbp < 80), 
                  (sbp >= 130) & (sbp < 139) | (dbp >= 80) & (dbp < 89), 
                  (sbp >= 140) | (dbp >= 90)]
    choices1 = ["Regular", "Elevated", "Stage 1 Hypertension", "Stage 2 Hypertension"]
    merged["Hypertension Classification"] = np.select(conditions1, choices1, default="Regular")
    merged["Hypertensive"] = ((merged["SBP Merged"] >= 130) | (merged["DBP Merged"] >= 80)).astype(int)
    return merged

def chol_fix(merged):
    merged["Missing Cholesterol?"] = merged["Total Cholesterol (mg/dl)"].isna().astype(int).replace({1: "Missing", 0: "Not Missing"})
    merged["Total Cholesterol (mg/dl)"] = merged["Total Cholesterol (mg/dl)"].fillna(merged.groupby(["Gender"])["Total Cholesterol (mg/dl)"].transform("mean"))
    return merged

def sex_grouping(merged):
    stats = ['count', 'mean', 'std']
    sex_table = merged.groupby('Gender')[['Age', 'SBP Merged', 'DBP Merged', 'Total Cholesterol (mg/dl)', 'Hypertensive']].agg({'Age': stats, 'SBP Merged': stats, 'DBP Merged': stats, 'Total Cholesterol (mg/dl)': stats, "Hypertensive": 'mean'})
    sex_table.to_csv('sex_table.csv', index='False')
    return merged, sex_table

def age_grouping(merged):
    bins = [14, 19, 25, 40, 55, 65, 80]
    labels = ["Teenager", "Young Adult", "Adult", "Older Adult", "Pre-Elderly", "Elderly"]
    merged["Age Group"] = pd.cut(merged["Age"], bins=bins, labels=labels)
    return merged

def hypertension_bar(merged):
    prev_df = merged.groupby(['Age Group', 'Gender'], observed=False)['Hypertensive'].mean().reset_index()
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=prev_df, x='Age Group', y='Hypertensive', hue='Gender', ax=ax)
    for container in ax.containers: ax.bar_label(container, fmt='%.1f%%', padding=3, labels=[f'{x*100:.1f}%' for x in container.datavalues])
    ax.set_title('Hypertension Prevalence by Sex and Age Group')
    ax.set_ylabel('Prevalence (Proportion)')
    ax.set_xlabel('Age Group')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('hypertension_prevalence_by_sex_age.png', dpi=300)
    plt.close()

def sbp_distribution(merged):
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.histplot(data=merged, x='SBP Merged', kde=True, color='skyblue', ax=ax)
    thresholds = {120: 'Elevated', 130: 'Stage 1 HTN', 140: 'Stage 2 HTN'}
    colors = ['green', 'orange', 'red']
    for (thresh, label), col in zip(thresholds.items(), colors): ax.axvline(x=thresh, color=col, linestyle='--', linewidth=1.5, label=f'{label} ({thresh} mmHg)')
    ax.set_title('Distribution of Systolic Blood Pressure with Clinical Thresholds')
    ax.set_xlabel('Systolic Blood Pressure (mmHg)')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.savefig('systolic_bp_distribution.png', dpi=300)
    plt.close()

def mean_cholesterol(merged):
    chol_stats = merged.groupby('Race-Ethnicity')['Total Cholesterol (mg/dl)'].agg(['mean', 'sem']).reset_index()
    chol_stats = chol_stats.sort_values(by='mean', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(y=chol_stats['Race-Ethnicity'], width=chol_stats['mean'], xerr=chol_stats['sem'], color='lightcoral', edgecolor='black', capsize=5)
    ax.bar_label(bars, fmt='%.1f', padding=10)
    ax.set_xlim(0, chol_stats['mean'].max() + 30)
    ax.set_title('Mean Total Cholesterol by Racial/Ethnic Group (with SEM)')
    ax.set_xlabel('Mean Total Cholesterol (mg/dL)')
    ax.set_ylabel('Racial/Ethnic Group')
    plt.tight_layout()
    plt.savefig('cholesterol_by_race.png', dpi=300)
    plt.close()

def age_bp(merged):
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.scatterplot(data=merged, x='Age', y='SBP Merged', hue='Gender', alpha=0.4, ax=ax)
    for gender, group in merged.groupby('Gender'): 
        clean_group = group[['Age', 'SBP Merged']].dropna()
        m, b = np.polyfit(clean_group['Age'], clean_group['SBP Merged'], 1)
        sns.regplot(data=clean_group, x='Age', y='SBP Merged', scatter=False, ax=ax, label=f'{gender} Line: SBP = {m:.2f}*(Age) + {b:.1f}')
    ax.set_title('Systolic Blood Pressure vs. Age by Gender')
    ax.set_xlabel('Age (Years)')
    ax.set_ylabel('Systolic Blood Pressure (mmHg)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('systolic_bp_vs_age.png', dpi=300)
    plt.close()

def bp_sex_crosstab(merged):
    crosstab_bp_sex = pd.crosstab(index=merged['Hypertension Classification'], columns=merged['Gender'], normalize='index') * 100
    print("Crosstab of BP Stage by Sex (Row Percentages):")
    print(crosstab_bp_sex.round(2).map(lambda x: f"{x:.2f}%"))

def data_prep():
    demo, bp, chol = load()
    info(demo, bp, chol)
    merged = merge(demo, bp, chol)
    merged = merge_bp(merged)
    merged = reorg(merged)
    merged = age_filtering(merged)
    merged = bp_grouping(merged)
    merged = chol_fix(merged)
    return merged

def data_visualization(merged):
    merged, sex_table = sex_grouping(merged)
    merged = age_grouping(merged)
    hypertension_bar(merged)
    sbp_distribution(merged)
    mean_cholesterol(merged)
    age_bp(merged)
    bp_sex_crosstab(merged)

merged = data_prep()
data_visualization(merged)
merged.to_csv('FinalData.csv', index='False')

Demographics: (9254, 46) with columns Index(['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN',
       'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC',
       'DMDBORN4', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDMARTL',
       'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY',
       'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'DMDHHSIZ',
       'DMDFMSIZ', 'DMDHHSZA', 'DMDHHSZB', 'DMDHHSZE', 'DMDHRGND', 'DMDHRAGZ',
       'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU',
       'SDMVSTRA', 'INDHHIN2', 'INDFMIN2', 'INDFMPIR'],
      dtype='str')
Blood Pressure: (8704, 21) with columns Index(['SEQN', 'PEASCCT1', 'BPXCHR', 'BPAARM', 'BPACSZ', 'BPXPLS', 'BPXPULS',
       'BPXPTY', 'BPXML1', 'BPXSY1', 'BPXDI1', 'BPAEN1', 'BPXSY2', 'BPXDI2',
       'BPAEN2', 'BPXSY3', 'BPXDI3', 'BPAEN3', 'BPXSY4', 'BPXDI4', 'BPAEN4'],
      dtype='str')
Cholesterol: (7435, 3) with columns Index(['SEQN', 